# Finding host galaxies from Rubin alerts

In this notebook we will find host galaxies for APDB alerts.

1. Select a region of the sky defined by the Deep Drilling fields.
2. Select DP2 objects with a filter on extendendedness, to capture galaxies.
3. Crossmatch the alerts with the DP2 objects.
3. Calculate the (u-g, g-r) differences and plot the respective color diagram.

In [ ]:
import lsdb

from functools import reduce
from hats.pixel_math import region_to_moc
from hats.inspection.visualize_catalog import plot_moc

from dask.distributed import Client

from lsdb.core.search.region_search import MOCSearch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

First, let's open the alert archive catalog:

In [ ]:
alerts = lsdb.open_catalog(
    "/sdf/data/rubin/shared/lsdb_commissioning/alert_archive", columns=["diaSourceId", "ra", "dec"]
)
alerts

Let's select a region in the sky defined by the Deep Drilling Fields:

In [ ]:
fields = {
    "ELAISS1": (9.45, -44.02),  # ELAIS-S1
    "XMM_LSS": (35.57, -4.82),  # XMM Large Scale Structure
    "ECDFS": (52.98, -28.12),  # Extended Chandra Deep Field South
    "COSMOS": (150.11, 2.23),  # COSMOS
    "EDFS_a": (58.9, -49.32),  # Euclid Deep Field South a
    "EDFS_b": (63.6, -47.60),  # Euclid Deep Field South b
}

cone_mocs = [
    # Using a 4 degree cone around each center.
    region_to_moc.cone_to_moc(ra=ra, dec=dec, radius_arcsec=4 * 3600, max_depth=10)
    for ra, dec in fields.values()
]
cone_moc = reduce(lambda a, b: a.union(b), cone_mocs)
plot_moc(cone_moc)

Let's select the flag columns for filtering:

In [ ]:
flag_cols = [f"{band}_pixelFlags_bad" for band in "ugr"]
flag_cols

As well as the extendedness and magnitude information:

In [ ]:
columns = ["objectId", "refExtendedness", "refSizeExtendedness"]
columns += [f"{band}_psfMag" for band in "ugr"]
columns += [f"{band}_psfMagErr" for band in "ugr"]
columns += flag_cols
columns

In [ ]:
filters = [("refExtendedness", ">", 0.7), ("refSizeExtendedness", ">", 0.7)]
filters += [(f"{band}_psfMagErr", "<", 0.05) for band in "ugr"]
filters

We can now open the DP2 collection. We will filter for galaxies and select the objects per-band magnitude information.

In [ ]:
dp2_galaxies = lsdb.open_catalog(
    "/sdf/data/rubin/shared/lsdb_commissioning/hats/dp2_rc/object_collection",
    # Select extendendness and per-band magnitude columns
    columns=columns,
    # Select the Deep Drilling Fields
    search_filter=MOCSearch(cone_moc),
    # Filter by large extendedness to avoid point sources and artifacts
    filters=filters,
)
dp2_galaxies

Let's remove any of the galaxies with at least one of the selected flags set:

In [ ]:
dp2_galaxies = dp2_galaxies.query("not (" + " or ".join(c for c in flag_cols) + ")")
dp2_galaxies

We then crossmatch the alerts to the DP2 data to find their galaxy hosts:

In [ ]:
matches = alerts.crossmatch(
    dp2_galaxies, n_neighbors=3, radius_arcsec=5, min_radius_arcsec=0.02, suffix_method="overlapping_columns"
)
matches

The last step is to calculate the per-band color information for each object in the resulting partitions:

In [ ]:
def color_diff(df):
    return {"u-g": df["u_psfMag"] - df["g_psfMag"], "g-r": df["g_psfMag"] - df["r_psfMag"]}


result = matches.map_partitions(color_diff, meta={"u-g": float, "g-r": float})
result

Let's setup a Dask Client to more effectively use the hardware available on this RSP machine (assuming a large 32GiB RAM instance):

In [ ]:
with Client(n_workers=4, memory_limit="8GiB"):
    df = result.compute()

Finally, let's plot the resulting color-color diagram:

In [ ]:
ax = plt.subplot()
h = ax.hist2d(df["u-g"], df["g-r"], bins=500, cmin=1, norm=LogNorm())
ax.set_xlabel("u-g")
ax.set_ylabel("g-r")
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-1, 2.5)
plt.colorbar(h[3], label="Number of galaxies")
plt.title("Color-color diagram")
plt.show()

## About

**Authors:** Sandro Campos, Konstantin Malanchev

**Last updated/verified on:** Jul 16, 2026

If you use `lsdb` for published research, please cite following [instructions](https://docs.lsdb.io/en/stable/citation.html).